# 01. Data Understanding & Exploratory Data Analysis (EDA)

This notebook **only explores** the Cupid Matchmaking Dataset (100,000 profile pairs): structure, distributions, the target `compatible`, correlations and demographic patterns.

It is **read-only**: the raw data is never modified, cleaned, encoded, split or saved here. Those tasks belong to the other notebooks (`02_preprocessing`, `03_logistic_regression`, ...).

## Step 0: Setup and Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RAW_PATH  = '../data/raw/cupid_algorithm_dataset.csv'
TARGET    = 'compatible'
LONGEVITY = 'relationship_longevity_months'


def text_columns(frame):
    """Columns that are neither numeric nor boolean (text / categorical)."""
    return [c for c in frame.columns
            if not pd.api.types.is_numeric_dtype(frame[c]) and not pd.api.types.is_bool_dtype(frame[c])]

In [2]:
df = pd.read_csv(RAW_PATH)
print(f'Dataset Shape: {df.shape}')
df.head()

Dataset Shape: (100000, 30)


## 1. Structure of the dataset

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   pair_id                        100000 non-null  int64  
 1   a_age                          100000 non-null  int64  
 2   a_education                    100000 non-null  int64  
 3   a_location                     100000 non-null  str    
 4   a_career_field                 100000 non-null  str    
 5   a_career_ambition              100000 non-null  float64
 6   a_openness                     100000 non-null  float64
 7   a_extraversion                 100000 non-null  float64
 8   a_agreeableness                100000 non-null  float64
 9   a_conscientiousness            100000 non-null  float64
 10  a_chronotype                   100000 non-null  float64
 11  a_spontaneity                  100000 non-null  float64
 12  a_love_language                100000 non-

In [4]:
df.describe().T

In [5]:
# Column groups (used by the plots below; df itself is not modified)
categorical  = text_columns(df)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
flag_cols    = [c for c in numeric_cols if set(df[c].dropna().unique()) <= {0, 1}]
continuous   = [c for c in numeric_cols if c not in flag_cols]
age_cols     = [c for c in ['a_age', 'b_age'] if c in df.columns]

print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print(f'Continuous/numeric: {len(continuous)} | 0/1 flags: {len(flag_cols)} | Categorical (text): {len(categorical)}')
print('Categorical columns:', categorical)

# Working copy for exploration-only columns (age categories, age gap)
eda = df.copy()

Rows: 100,000, Columns: 30
Continuous/numeric: 23 | 0/1 flags: 0 | Categorical (text): 6
Categorical columns: ['a_location', 'a_career_field', 'a_love_language', 'b_location', 'b_career_field', 'b_love_language']


## 2. Target variable distribution

In [6]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET, hue=TARGET, palette='Set2', legend=False)
plt.title('Target Variable Distribution (compatible)')
plt.xlabel('Compatible (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

print(df[TARGET].value_counts(normalize=True))

compatible
0    0.57119
1    0.42881
Name: proportion, dtype: float64


## 3. Distribution of the features (independent exploration)

In [7]:
cols_to_plot = continuous[:12]
n = len(cols_to_plot)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
    for ax, col in zip(np.ravel(axes), cols_to_plot):
        sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax)
        ax.set_title(col)
    for ax in np.ravel(axes)[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [8]:
# Categorical features: share of each category
for col in categorical:
    if df[col].nunique() <= 15:
        print(f'\n{col}:')
        print((df[col].value_counts(normalize=True, dropna=False) * 100).round(1).astype(str) + ' %')
    else:
        print(f'\n{col}: {df[col].nunique()} unique values')


a_location:
a_location
Urban       49.8 %
Suburban    35.0 %
Rural       15.2 %
Name: proportion, dtype: str

a_career_field:
a_career_field
Finance             10.2 %
Engineering         10.0 %
Education           10.0 %
Law                 10.0 %
Creative Arts       10.0 %
Entrepreneurship    10.0 %
Tech                10.0 %
Healthcare          10.0 %
Marketing           10.0 %
Science              9.9 %
Name: proportion, dtype: str

a_love_language:
a_love_language
Words of Affirmation    20.2 %
Quality Time            20.1 %
Receiving Gifts         20.0 %
Physical Touch          19.9 %
Acts of Service         19.9 %
Name: proportion, dtype: str

b_location:
b_location
Urban       49.8 %
Suburban    35.1 %
Rural       15.1 %
Name: proportion, dtype: str

b_career_field:
b_career_field
Marketing           10.1 %
Science             10.1 %
Creative Arts       10.1 %
Law                 10.0 %
Engineering         10.0 %
Entrepreneurship    10.0 %
Healthcare          10.0 %
Finance   

## 4. Age analysis

In [9]:
bins   = [0, 2, 12, 19, 35, 60, 150]
labels = ['Newborn', 'Child', 'Teenager', 'Young Adult', 'Senior Adult', 'Senior']
age_cat_cols = []
for col in age_cols:
    new = f'{col}_category'
    eda[new] = pd.cut(eda[col], bins=bins, labels=labels, right=False)
    age_cat_cols.append(new)

if {'a_age', 'b_age'} <= set(eda.columns):
    eda['age_gap'] = (eda['a_age'] - eda['b_age']).abs()

if age_cat_cols:
    fig, axes = plt.subplots(1, len(age_cat_cols), figsize=(6 * len(age_cat_cols), 4), squeeze=False)
    for ax, col in zip(axes[0], age_cat_cols):
        eda[col].value_counts().reindex(labels).plot(kind='bar', ax=ax, title=f'{col} distribution')
    plt.tight_layout()
    plt.show()
else:
    print('No age column found.')

## 5. Relationship longevity vs compatibility

In [10]:
if LONGEVITY in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=TARGET, y=LONGEVITY, hue=TARGET, palette='Set2', legend=False)
    plt.title('Relationship Longevity (Months) by Compatibility')
    plt.show()
else:
    print(f'Column {LONGEVITY} not found.')

## 6. Feature correlations

In [11]:
corr_cols = (numeric_cols + [TARGET])[:25]
corr = df[corr_cols].corr()
plt.figure(figsize=(max(8, len(corr_cols) * 0.55), max(6, len(corr_cols) * 0.45)))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=len(corr_cols) <= 14, fmt='.2f')
plt.title('Correlation matrix of numeric features')
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
high = pairs[pairs.abs() > 0.9]
print('Highly correlated pairs (|r| > 0.9):', high.to_dict() if len(high) else 'none')

Highly correlated pairs (|r| > 0.9): none


In [12]:
numeric_df = df.select_dtypes(include=[np.number])
corrs = numeric_df.corr()[TARGET].sort_values(ascending=False)
print('Top Correlations with Target (compatible):')
print(corrs)

top = corrs.drop(TARGET).abs().sort_values(ascending=False).head(15).index
plt.figure(figsize=(8, 5))
corrs.drop(TARGET).loc[top].sort_values().plot(kind='barh')
plt.title('Top 15 correlations with the target')
plt.show()

Top Correlations with Target (compatible):
compatible                       1.000000
compatibility_score              0.794533
relationship_longevity_months    0.438856
a_agreeableness                  0.006972
b_spontaneity                    0.004947
a_extraversion                   0.004131
a_education                      0.003618
b_conscientiousness              0.002938
a_career_ambition                0.002838
a_chronotype                     0.001959
b_chronotype                     0.001713
a_age                            0.000938
a_openness                       0.000926
b_education                     -0.000234
b_extraversion                  -0.000266
a_emotional_expressiveness      -0.000378
pair_id                         -0.000852
a_conscientiousness             -0.001163
a_spontaneity                   -0.001745
b_agreeableness                 -0.002195
b_career_ambition               -0.003730
b_openness                      -0.004942
b_emotional_expressiveness      -

## 7. Compatibility by group

As a reference, the overall compatibility rate is compared with each group; only differences larger than 5 points are printed.

In [13]:
base_rate = eda[TARGET].mean()
print(f'Overall compatibility rate: {base_rate:.1%}\n')

group_cols = [c for c in categorical + age_cat_cols if eda[c].nunique() <= 15]
found = False
for col in group_cols:
    rate = eda.groupby(col, observed=True)[TARGET].mean()
    if rate.max() - rate.min() > 0.05:
        found = True
        print(f'{col}:')
        print((rate * 100).round(1).astype(str) + ' %\n')
if not found:
    print('No group shows a difference larger than 5 points.')

for col in age_cat_cols:
    rate = eda.groupby(col, observed=True)[TARGET].mean().reindex(labels).dropna()
    rate.plot(kind='bar', title=f'Compatibility rate by {col}', figsize=(7, 4))
    plt.ylabel('Compatibility rate')
    plt.show()

Overall compatibility rate: 42.9%

b_career_field:
b_career_field
Creative Arts       39.8 %\n
Education           42.9 %\n
Engineering         44.1 %\n
Entrepreneurship    45.3 %\n
Finance             42.7 %\n
Healthcare          41.8 %\n
Law                 40.7 %\n
Marketing           42.7 %\n
Science             44.2 %\n
Tech                44.6 %\n
Name: compatible, dtype: str


In [14]:
# Mean of each continuous feature for compatible vs. not compatible pairs
means = df.groupby(TARGET)[continuous].mean().T
std = df[continuous].std().replace(0, np.nan)
means['std_diff'] = (means.get(1, np.nan) - means.get(0, np.nan)) / std
display(means.sort_values('std_diff', key=lambda s: s.abs(), ascending=False).head(15))

compatible                             0          1  std_diff
compatibility_score            55.752224  72.939437  1.605414
relationship_longevity_months  60.793904  79.712180  0.886743
a_agreeableness                 0.500045   0.503198  0.014087
b_age                          36.575815  36.435251 -0.012849
b_emotional_expressiveness      0.501375   0.498892 -0.010991
b_spontaneity                   0.499354   0.501617  0.009997
b_openness                      0.500854   0.498622 -0.009986
a_extraversion                  0.499326   0.501202  0.008347
b_career_ambition               0.496624   0.495310 -0.007537
a_education                     2.948476   2.956647  0.007311
b_conscientiousness             0.499495   0.500821  0.005937
a_career_ambition               0.496033   0.497036  0.005734
b_agreeableness                 0.500525   0.499535 -0.004435
a_chronotype                    0.499315   0.500205  0.003958
a_spontaneity                   0.500354   0.499556 -0.003527


## 8. Interaction effects

In [15]:
# Age gap vs compatibility
if 'age_gap' in eda.columns:
    gap_bins = pd.cut(eda['age_gap'], bins=[-0.1, 2, 5, 10, 20, 200],
                      labels=['0-2', '3-5', '6-10', '11-20', '20+'])
    gap_rate = eda.groupby(gap_bins, observed=True)[TARGET].mean()
    gap_rate.plot(kind='bar', title='Compatibility rate by age gap (years)', figsize=(7, 4))
    plt.ylabel('Compatibility rate')
    plt.show()
    print((gap_rate * 100).round(1).astype(str) + ' %')
else:
    print('No age gap available (columns a_age / b_age not found).')

age_gap
0-2      47.0 %
3-5      45.9 %
6-10     43.4 %
11-20    41.0 %
20+      40.6 %
Name: compatible, dtype: str


In [16]:
# Both partners older than 35 vs both younger
if {'a_age', 'b_age'} <= set(eda.columns):
    both_old   = eda[(eda['a_age'] > 35) & (eda['b_age'] > 35)][TARGET].mean()
    both_young = eda[(eda['a_age'] <= 35) & (eda['b_age'] <= 35)][TARGET].mean()
    mixed      = eda[((eda['a_age'] > 35) ^ (eda['b_age'] > 35))][TARGET].mean()
    print(f'Compatibility, both > 35 : {both_old:.1%}')
    print(f'Compatibility, both <= 35: {both_young:.1%}')
    print(f'Compatibility, mixed     : {mixed:.1%}')

Compatibility, both > 35 : 43.9%
Compatibility, both <= 35: 45.0%
Compatibility, mixed     : 41.4%


In [17]:
# Profile comparison: standardized means of the top features
top_feats = means['std_diff'].abs().sort_values(ascending=False).head(10).index.tolist()
if top_feats:
    z = (df[continuous] - df[continuous].mean()) / df[continuous].std().replace(0, np.nan)
    comp = z[top_feats].groupby(df[TARGET]).mean().T
    comp.columns = ['Not compatible' if c == 0 else 'Compatible' for c in comp.columns]
    comp.plot(kind='bar', figsize=(11, 5))
    plt.title('Standardized feature means by compatibility')
    plt.ylabel('Mean (z-score)')
    plt.tight_layout()
    plt.show()

## Summary

This notebook gave an overview of the dataset: its structure, the balance of the target `compatible`, the distribution of the features, their correlations with the target and a few group and interaction effects. Everything that modifies or models the data (quality treatment, encoding, train/test split, scaling, models) is done in the following notebooks.